In [3]:
def edit_distance(a: str, b: str) -> int:
    m, n = len(a), len(b)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # base cases
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    # fill table
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # delete
                    dp[i][j-1],    # insert
                    dp[i-1][j-1]   # replace
                )

    return dp[m][n]


# tests
print(edit_distance("cat", "cut"))       # 1 (replace a→u)
print(edit_distance("kitten", "sitting")) # 3
print(edit_distance("", "hello"))         # 5 (5 inserts)
print(edit_distance("abc", "abc"))        # 0 (identical)
print(edit_distance("rat", "cat"))           # 3 (3 deletes)

1
3
5
0
1


In [6]:
vocab = ["i", "like", "cats", "hate", "dogs"]

# Transition: P(next_word | current_word)
transitions = {
    "i":    {"like": 0.6,  "hate": 0.3,  "cats": 0.05, "dogs": 0.05},
    "like": {"cats": 0.5,  "dogs": 0.4,  "i":    0.05, "hate": 0.05},
    "hate": {"cats": 0.5,  "dogs": 0.4,  "i":    0.05, "like": 0.05},
    "cats": {"i":    0.5,  "like": 0.2,  "hate": 0.2,  "dogs": 0.1 },
    "dogs": {"i":    0.5,  "like": 0.2,  "hate": 0.2,  "cats": 0.1 },
}

# Emission: P(observed_typo | intended_word)
emissions = {
    "i":    {"i":    0.9,  "ii":    0.05, "u":     0.05},
    "like": {"like": 0.8,  "lke":   0.1,  "likee": 0.07, "lyke":  0.03},
    "cats": {"cats": 0.8,  "catz":  0.1,  "kats":  0.07, "catss": 0.03},
    "hate": {"hate": 0.8,  "hte":   0.1,  "haet":  0.07, "h8":    0.03},
    "dogs": {"dogs": 0.8,  "dogz":  0.1,  "digs":  0.07, "doggs": 0.03},
}

# Start probabilities
start = {"i": 0.7, "like": 0.05, "cats": 0.1, "hate": 0.1, "dogs": 0.05}


# ---------- Viterbi ----------------------------------------

def viterbi(observed_words, verbose=True):
    T = len(observed_words)

    # dp[t][s]      = best probability of any path ending in state s at step t
    # backptr[t][s] = which previous state led to that best probability
    dp      = [{} for _ in range(T)]
    backptr = [{} for _ in range(T)]

    # --- Initialisation (t = 0) ---
    for s in vocab:
        emit       = emissions[s].get(observed_words[0], 1e-9)
        dp[0][s]   = start[s] * emit
        backptr[0][s] = None

    if verbose:
        print(f"t=0  obs='{observed_words[0]}'")
        for s in vocab:
            print(f"       {s:<6}  start={start[s]:.2f}  emit={emissions[s].get(observed_words[0], 1e-9):.4f}"
                  f"  dp={dp[0][s]:.2e}")

    # --- Recursion (t = 1 … T-1) ---
    for t in range(1, T):
        obs = observed_words[t]
        if verbose:
            print(f"\nt={t}  obs='{obs}'")

        for s in vocab:
            emit = emissions[s].get(obs, 1e-9)
            best_prob, best_prev = -1, None

            for prev in vocab:
                trans = transitions[prev].get(s, 1e-9)
                p     = dp[t-1][prev] * trans * emit
                if p > best_prob:
                    best_prob, best_prev = p, prev

            dp[t][s]      = best_prob
            backptr[t][s] = best_prev

            if verbose:
                print(f"       {s:<6}  best_prev={best_prev:<6}  emit={emit:.4f}  dp={best_prob:.2e}")

    # --- Backtracking ---
    best_last = max(dp[T-1], key=dp[T-1].get)
    path = [best_last]
    for t in range(T-1, 0, -1):
        path.insert(0, backptr[t][path[0]])

    return path


# ---------- Run examples ------------------------------------

test_sentences = [
    "i lke catz",
    "i hte dogz",
    "i lyke dogs",
    "i haet catss",
    "u lke doggs",
]

for sentence in test_sentences:
    observed = sentence.strip().lower().split()
    print("=" * 55)
    print(f"INPUT  : {' '.join(observed)}")
    print("-" * 55)
    path = viterbi(observed, verbose=True)
    print("-" * 55)
    print(f"OUTPUT : {' '.join(path)}")
    print()

INPUT  : i lke catz
-------------------------------------------------------
t=0  obs='i'
       i       start=0.70  emit=0.9000  dp=6.30e-01
       like    start=0.05  emit=0.0000  dp=5.00e-11
       cats    start=0.10  emit=0.0000  dp=1.00e-10
       hate    start=0.10  emit=0.0000  dp=1.00e-10
       dogs    start=0.05  emit=0.0000  dp=5.00e-11

t=1  obs='lke'
       i       best_prev=i       emit=0.0000  dp=6.30e-19
       like    best_prev=i       emit=0.1000  dp=3.78e-02
       cats    best_prev=i       emit=0.0000  dp=3.15e-11
       hate    best_prev=i       emit=0.0000  dp=1.89e-10
       dogs    best_prev=i       emit=0.0000  dp=3.15e-11

t=2  obs='catz'
       i       best_prev=like    emit=0.0000  dp=1.89e-12
       like    best_prev=like    emit=0.0000  dp=3.78e-20
       cats    best_prev=like    emit=0.1000  dp=1.89e-03
       hate    best_prev=like    emit=0.0000  dp=1.89e-12
       dogs    best_prev=like    emit=0.0000  dp=1.51e-11
--------------------------------------